In [ ]:
# Source of the DS: https://archive.ics.uci.edu/dataset/577/codon+usage
# The target is Kingdom, it classifies organisms into all 11 biological kingdoms.

# LIST OF TASKS:
# Identify original problems again
# Try sampling method
# Try classification synthetic data method: SMOTE / ADASYN / CTGAN
# Try noise management method
# Try overlap management method
# Try outlier management method
# Re analyze the dataset
# Compare original vs optimized
# Evaluate balance and realism

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from collections import Counter
from scipy.stats import median_abs_deviation

from imblearn.over_sampling import RandomOverSampler, SMOTENC, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import KNeighborsClassifier

<h3><center>Understanding and Exploring the Dataset<h3>

In [3]:
# Loading the ds, using low_memory=False because UUU and UUC have mixed types
df = pd.read_csv('datasets/codon_usage.csv', low_memory=False)
df.head()

,Kingdom,DNAtype,SpeciesID,Ncodons,SpeciesName,UUU,UUC,UUA,UUG,CUU,...,CGG,AGA,AGG,GAU,GAC,GAA,GAG,UAA,UAG,UGA
0,vrl,0,100217,1995,Epizootic haematopoietic necrosis virus,0.01654,0.01203,0.00050,0.00351,0.01203,...,0.00451,0.01303,0.03559,0.01003,0.04612,0.01203,0.04361,0.00251,0.00050,0.00000
1,vrl,0,100220,1474,Bohle iridovirus,0.02714,0.01357,0.00068,0.00678,0.00407,...,0.00136,0.01696,0.03596,0.01221,0.04545,0.01560,0.04410,0.00271,0.00068,0.00000
2,vrl,0,100755,4862,Sweet potato leaf curl virus,0.01974,0.0218,0.01357,0.01543,0.00782,...,0.00596,0.01974,0.02489,0.03126,0.02036,0.02242,0.02468,0.00391,0.00000,0.00144
3,vrl,0,100880,1915,Northern cereal mosaic virus,0.01775,0.02245,0.01619,0.00992,0.01567,...,0.00366,0.01410,0.01671,0.03760,0.01932,0.03029,0.03446,0.00261,0.00157,0.00000
4,vrl,0,100887,22831,Soil-borne cereal mosaic virus,0.02816,0.01371,0.00767,0.03679,0.01380,...,0.00604,0.01494,0.01734,0.04148,0.02483,0.03359,0.03679,0.00000,0.00044,0.00131


In [4]:
df.info()
# that is a huge ds

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13028 entries, 0 to 13027
Data columns (total 69 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Kingdom      13028 non-null  object 
 1   DNAtype      13028 non-null  int64  
 2   SpeciesID    13028 non-null  int64  
 3   Ncodons      13028 non-null  int64  
 4   SpeciesName  13028 non-null  object 
 5   UUU          13028 non-null  object 
 6   UUC          13028 non-null  object 
 7   UUA          13028 non-null  float64
 8   UUG          13028 non-null  float64
 9   CUU          13028 non-null  float64
 10  CUC          13028 non-null  float64
 11  CUA          13028 non-null  float64
 12  CUG          13028 non-null  float64
 13  AUU          13028 non-null  float64
 14  AUC          13028 non-null  float64
 15  AUA          13028 non-null  float64
 16  AUG          13028 non-null  float64
 17  GUU          13028 non-null  float64
 18  GUC          13028 non-null  float64
 19  GUA 

In [4]:
# Let's check the duplicated rows
print(f'DS shape:',df.shape)
number_of_duplicated_rows =df.duplicated().sum()
print(f'Number of duplicated rows:{number_of_duplicated_rows}')

DS shape: (13028, 69)
Number of duplicated rows:0


In [5]:
df['UUU']=pd.to_numeric(df['UUU'],errors='coerce')
df['UUC'] = pd.to_numeric(df['UUC'],errors='coerce')

# The DS is big, let's check the unique values in each column.
for column in df.columns:
        print(f"Unique values in {column}: {df[column].unique()}")
# Based on ds description: DNAtype is used to identify the genomic origin of a sequence.

Unique values in Kingdom: ['vrl' 'arc' 'bct' 'phg' 'plm' 'pln' 'inv' 'vrt' 'mam' 'rod' 'pri']
Unique values in DNAtype: [ 0  6  4  2  1  3  7  9  5 11 12]
Unique values in SpeciesID: [100217 100220 100755 ...   9601   9602   9606]
Unique values in Ncodons: [    1995     1474     4862 ...    96254 40662582  8998998]
Unique values in SpeciesName: ['Epizootic haematopoietic necrosis virus' 'Bohle iridovirus'
 'Sweet potato leaf curl virus' ...
 'mitochondrion Pongo pygmaeus pygmaeus' 'Homo sapiens'
 'mitochondrion Homo sapiens']
Unique values in UUU: [0.01654 0.02714 0.01974 ... 0.02314 0.01791 0.00554]
Unique values in UUC: [0.01203 0.01357 0.0218  ... 0.03789 0.03215 0.03555]
Unique values in UUA: [0.0005  0.00068 0.01357 ... 0.03671 0.02285 0.04059]
Unique values in UUG: [0.00351 0.00678 0.01543 ... 0.0004  0.00491 0.00619]
Unique values in CUU: [0.01203 0.00407 0.00782 ... 0.00504 0.0299  0.03415]
Unique values in CUC: [0.03208 0.02849 0.01111 ... 0.04792 0.05322 0.05042]
Unique value

In [6]:
df  = df.dropna(subset=['UUU', 'UUC'])
# let's check, do I have columns with only one unique value
unique_values = df.columns[df.nunique() == 1]
unique_values

Index([], dtype='object')

In [7]:
df["Kingdom"].value_counts()

Kingdom
bct    2919
vrl    2831
pln    2523
vrt    2077
inv    1345
mam     572
phg     220
rod     215
pri     180
arc     126
plm      18
Name: count, dtype: int64

In [8]:
# Kingdom colum shows biological groups,
# bct = bacteria, vrl = virus
# pln = plant
# inv = invertebrate, mam = mammal 
# To understand better those abbreviations, I used QWEN.

In [10]:
df['DNAtype'].value_counts()

DNAtype
0     9265
1     2899
2      816
4       31
12       5
3        2
9        2
5        2
11       2
6        1
7        1
Name: count, dtype: int64

In [11]:
summary = df.describe().T
summary

,count,mean,std,min,25%,50%,75%,max
DNAtype,13026.0,0.367265,0.688764,0.0,0.00000,0.000000,1.000000,1.200000e+01
SpeciesID,13026.0,130443.036926,124777.067741,7.0,28851.25000,81971.500000,222890.500000,4.653640e+05
Ncodons,13026.0,79617.758483,719755.572570,1000.0,1602.00000,2929.000000,9120.000000,4.066258e+07
UUU,13026.0,0.024818,0.017628,0.0,0.01391,0.021750,0.031308,2.173000e-01
UUC,13026.0,0.023440,0.011598,0.0,0.01538,0.021905,0.029210,9.169000e-02
...,...,...,...,...,...,...,...,...
GAA,13026.0,0.028291,0.014343,0.0,0.01736,0.026085,0.036800,1.448900e-01
GAG,13026.0,0.021683,0.015019,0.0,0.00971,0.020540,0.031128,1.585500e-01
UAA,13026.0,0.001640,0.001785,0.0,0.00056,0.001380,0.002370,4.520000e-02
UAG,13026.0,0.000590,0.000882,0.0,0.00000,0.000420,0.000830,2.561000e-02


In [9]:
# UAG, UAA, and UGA are very close to zero, when I asked GPT, the response was: 
# These are condon, which instruction for cell to adds one amino acid for building  protein. 
# When it is very low that means, STOP building the chain. 

In [10]:
# Having ID number, Species name  and Ncodons are cheat code for model.
df = df.drop(columns=['SpeciesID','SpeciesName', 'Ncodons'])

In [11]:
# let's check can I find any useful information from those columns and kingdom column
stoped_condons = ['UAG', 'UAA', 'UGA']
for column in stoped_condons:
    grouped = df.groupby("Kingdom")[column].quantile([0.5, 0.95, 0.99]).unstack().round(4)
    print(f"\n{column}:")
    display(grouped)
# The gap between 0.5 and 0.95 is very big, but between 0.95 and 0.99 is not that big
# that means there are some outliers in the data, but not many.



UAG:


,0.50,0.95,0.99
Kingdom,,,
arc,0.0006,0.0015,0.0021
bct,0.0005,0.0014,0.0021
inv,0.0005,0.0024,0.0065
mam,0.0003,0.0019,0.0029
phg,0.0002,0.0010,0.0019
plm,0.0006,0.0009,0.0013
pln,0.0004,0.0018,0.0033
pri,0.0003,0.0018,0.0032
rod,0.0000,0.0014,0.0026



UAA:


,0.50,0.95,0.99
Kingdom,,,
arc,0.0013,0.0032,0.0038
bct,0.0013,0.0032,0.0046
inv,0.0018,0.0049,0.0190
mam,0.0009,0.0028,0.0034
phg,0.0024,0.0044,0.0050
plm,0.0013,0.0022,0.0030
pln,0.0010,0.0032,0.0049
pri,0.0016,0.0074,0.0087
rod,0.0006,0.0035,0.0093



UGA:


,0.50,0.95,0.99
Kingdom,,,
arc,0.0015,0.0028,0.0039
bct,0.0011,0.0034,0.0080
inv,0.0009,0.0257,0.0310
mam,0.0259,0.0316,0.0321
phg,0.0020,0.0034,0.0050
plm,0.0019,0.0027,0.0038
pln,0.0008,0.0028,0.0133
pri,0.0157,0.0282,0.0314
rod,0.0263,0.0316,0.0318


<h3><center> Feature Engineering <h3>

### Overlap between classes

In [12]:
# In this step, I am going to check: Are they target' calsses well separated? 
# If I look at the nearest sample in codon space, is it usually from the same kingdom?
# KNN gets training instances and predicts from nearby points
# with neighbors=1,  it uses the single nearest neighbor.

class_encoder = LabelEncoder()
X = df.drop(columns=['Kingdom'])
y = class_encoder.fit_transform(df['Kingdom'])

# DNAtype is catergorica column. 
continuous_columns = [column for column in X.columns if column != 'DNAtype']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# I am not gonna tune production model here
# I wanna a very local over lap check. 
Overlap_check = KNeighborsClassifier(1, n_jobs=-1).fit(X_train, y_train).predict(X_test)
# Source: https://scikit-learn.org/stable/modules/neighbors.html
print(f" KNN accuracy: {(Overlap_check == y_test).mean():.2f}")

#Making a for loop to check class by class. 
for numeric, name in enumerate(class_encoder.classes_):
    # Which test rows belong to class number? 
    m =y_test == numeric
    acc= (Overlap_check[m]== y_test[m]).mean()
    print(f" {name:4s} (Quantity = {m.sum():4d}): {acc:.3f}" + (" = can be overlapped" if acc < .7 else ""))
# The accuracy is acceptable.
# This Ds is very imbalanced, claases are not equally reliable. 
# KKN says that plm and pri are hard to separate. 
# For debugging in for loop, I used QWEN. 

 KNN accuracy: 0.93
 arc  (Quantity =   38): 0.816
 bct  (Quantity =  876): 0.957
 inv  (Quantity =  404): 0.832
 mam  (Quantity =  172): 0.890
 phg  (Quantity =   66): 0.773
 plm  (Quantity =    5): 0.400 = can be overlapped
 pln  (Quantity =  757): 0.959
 pri  (Quantity =   54): 0.630 = can be overlapped
 rod  (Quantity =   64): 0.781
 vrl  (Quantity =  849): 0.966
 vrt  (Quantity =  623): 0.968


d:\ML_DE\Advanced-Data-Analytics-\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "d:\ML_DE\Advanced-Data-Analytics-\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [13]:
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (9118, 65)
Shape of y_train: (9118,)
Shape of X_test: (3908, 65)
Shape of y_test: (3908,)


### Outlier detection and management

In [14]:
# To evaluate, I am gonna build a evaluation function.
# having same model and same test ds for all the evaluations.
# my ds is very imbalanced, so i need a consistent metrics.
def evaluation(X_train, y_train, X_test, y_test, label=""):
    random_forest_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
    predicted = random_forest_model.predict(X_test)
    f1 = f1_score(y_test, predicted, average='macro')
    print(f"{label:25s} F1={f1:.2f}")
    return f1, predicted

baseline_f1, predicted_base = evaluation(X_train, y_train, X_test, y_test, "Baseline")

Baseline                  F1=0.72


In [15]:
# I am going  build two outlier detection methods: IQR + Isolation Forest.
# This methods is recommended by Lecture.

# to not filer out  rare classes.  
train_counts = pd.Series(y_train).value_counts()
small_classes = train_counts[train_counts<60].index

q1, q3 = X_train[continuous_columns].quantile(0.25), X_train[continuous_columns].quantile(0.75)
iqr = q3- q1

# IQR method and Isolation Forest method are used together to detect outliers.
iqr_flag =((X_train[continuous_columns] < q1-1.5*iqr) | (X_train[continuous_columns] > q3 + 1.5 *iqr)).any(axis=1)
iso_flag = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1).fit_predict(X_train[continuous_columns]) == -1
drop_row =iqr_flag & iso_flag &~pd.Series(y_train).isin(small_classes).to_numpy()

X_train_clean = X_train.loc[~drop_row]
y_train_clean = y_train[~drop_row]

print(f"Removed {drop_row.sum()} rows = {drop_row.mean()*100:.1f}%")
outlier_f1, pred_outlier =evaluation(X_train_clean, y_train_clean, X_test, y_test, "After outlier removal")


Removed 183 rows = 2.0%
After outlier removal     F1=0.72


### Noise manangement

In [16]:
# After removing the some suspicious rows in previous cell. 
# do I still improve the model,  if I shrink extreme codon values instead of deleting more rows?
# But in real biological signal, those extreme values can be important, but noise managerment is mentioned in exercise pdf, so I am gonna try it.
# this step can be  useful only if the score clearly improves.

# calculating mad value for each feature.
mad= X_train_clean[continuous_columns].apply(median_abs_deviation)
med = X_train_clean[continuous_columns].median()
X_noise = X_train_clean.copy()
for column in continuous_columns:
    # if mad is zero, the feature does not have useful spread
    if mad[column] > 0:
        X_noise[column] = X_noise[column].clip(max(med[column]-5*mad[column], 0), med[column]+5*mad[column])
f1_noise, predicted_noise = evaluation(X_noise, y_train_clean, X_test, y_test, "After noise clipping")
# Source: https://www.blog.trainindata.com/winsorization-handling-outliers-in-machine-learning/
# F1 score is not improved, I skip this step.

After noise clipping      F1=0.72


### Overlap mangement 

In [17]:
# The target is very imbalanced, I am going to use TomekLinks which is under sampling method.
tomelink_method = TomekLinks(sampling_strategy='auto')
X_train_tome, y_train_tome = tomelink_method.fit_resample(X_train_clean, y_train_clean)
print(f"TomekLinks removed {len(y_train_clean)-len(y_train_tome)} boundary samples")
f1_tomek, _ = evaluation(X_train_tome, y_train_tome, X_test, y_test, "After TomekLinks")

TomekLinks removed 38 boundary samples
After TomekLinks          F1=0.73


### Sampling

In [18]:
# Using under sampling and over sampling methods.
# Evlauating both methods with evaluation function which I built before.

# RANDOM OVER SAMPLING:
random_over_sampler = RandomOverSampler(random_state=42)
X_ros, y_ros = random_over_sampler.fit_resample(X_train_clean, y_train_clean)
f1_ros, _ = evaluation(X_ros, y_ros, X_test, y_test, "Random Over Sampling")

# RANDOM UNDER SAMPLING:
random_under_sampler = RandomUnderSampler(random_state=42)
X_rus, y_rus = random_under_sampler.fit_resample(X_train_clean, y_train_clean)
f1_rus, _ = evaluation(X_rus, y_rus, X_test, y_test, "Random Under Sampling")
# I think under sampling threw away valuable data. 
# Over sampling gives more presences to the rare classes, that is why it got better results.

Random Over Sampling      F1=0.77
Random Under Sampling     F1=0.44


### Data Generating

In [19]:
# Generating synthetic minority samples:
# The ADASYN method is not stable for this class structure.

# Controlling the neighborhood used to create new samples.
k_neighbor = min(3, min(Counter(y_train_clean).values())-1)
# smotic can handle catergorical and continues features. 
dna_column = X_train_clean.columns.get_loc("DNAtype")
smotenc = SMOTENC(categorical_features=[dna_column],random_state=42,k_neighbors=k_neighbor)
X_smote, y_smote = smotenc.fit_resample(X_train_clean, y_train_clean)

f1_smote, _ = evaluation(X_smote, y_smote, X_test, y_test, "SMOTE")
# Source: https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html
# The F1 score shows that balancing the classes acrtually improved the model. 

SMOTE                     F1=0.78


In [20]:
# After all optimization steps in this cell, I am gonna check the data properties.

# UCI says:codon frequencies are normalized by total codon count.
# I asked: Google AI mode: What is the valid range for codon columns ?
# The response was:  1. Relative Frequency (0 to 1)
# In a standard frequency matrix where columns sum to 1, the valid range for any individual codon value is [0.0, 1.0]
# Based on that information, the codon columns almost always sum to 1


optimized_df = pd.DataFrame(X_smote, columns=X_train_clean.columns)
# this is a good method to check, Is the generated data still in normalized frequency structure? 
continuous_row_sums = optimized_df[continuous_columns].sum(axis=1)
print(f"Codon row sums mean: {continuous_row_sums.mean():.2f}")
# Class balance:
print(f"\nClass counts after smotenc: {dict(sorted(Counter(y_smote).items()))}")
# What I can see from the output, generated data is in normalized frequency structure.

Codon row sums mean: 1.00

Class counts after smotenc: {0: 1975, 1: 1975, 2: 1975, 3: 1975, 4: 1975, 5: 1975, 6: 1975, 7: 1975, 8: 1975, 9: 1975, 10: 1975}


<h3><center>Comparing original vs optimized<h3>

In [21]:
# After finding out that generating synthetic data is in normalized frequency structure.
# I am gonna seee:  pairwise relationship pattern between original and optimized data.
original_df = X_train.copy()
# Creating correlation matrix for oversampling and original data. 
original_df_correlation = original_df[continuous_columns].corr().to_numpy()
optimized_df_correlation = optimized_df[continuous_columns].corr().to_numpy()
upper = np.triu_indices_from(original_df_correlation, k=1)
# Computing between original and optimized correlation matrices.
# Close to 1 means the same pattern
# Close to 0 weak similarity.
correlation_similarity = np.corrcoef(original_df_correlation[upper], optimized_df_correlation[upper])[0, 1]

print(f"Correlation similarity: {correlation_similarity:.2f}")
# Source: https://stackoverflow.com/questions/73581384/plotting-a-fancy-diagonal-correlation-matrix-with-coefficients-in-upper-triangle
# They are quite similar, which is a good sign.

Correlation similarity: 0.96


In [22]:
# In this cell, I am gonna dive in deeper: 
# testing the pipeline by doing smotenc inside each fold
# Training  on the full resampled training ds and evaluating on X_test, which is untouched ds.
# And then showing which classes are improved?

# Balancing the training fold. 
pipe = ImbPipeline([('SMOTENC', SMOTENC(categorical_features=[dna_column], random_state=42, k_neighbors=k_neighbor)),
                  ('RandomForest', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])
# Cross validation
cv_scores = cross_val_score(pipe, X_train_clean, y_train_clean, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='f1_macro')
print(f"CV F1 = {cv_scores.mean():.2f}")

CV F1 = 0.79


In [23]:
final_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_smote, y_smote)
predicted_final = final_model.predict(X_test)
final_f1 = f1_score(y_test, predicted_final, average="macro")

# baseline comparison
print(f"F1: {final_f1:.2f} VS baseline: {baseline_f1:.2f}")

base_f1 = f1_score(y_test, predicted_base, average=None)
optimized_f1 = f1_score(y_test, predicted_final, average=None)

print(f"\n{'Class':4s} {'Base':>4s} {'Optimized':>8s}")
for name, base, optimized in zip(class_encoder.classes_, base_f1, optimized_f1):
    print(f"{name:4s} {base:6.4f} {optimized:6.4f})") 
# For debugging in for loop, I used QWEN.

F1: 0.78 VS baseline: 0.72

Class Base Optimized
arc  0.5660 0.7324)
bct  0.9393 0.9506)
inv  0.8033 0.8410)
mam  0.8261 0.8537)
phg  0.6186 0.8500)
plm  0.0000 0.0000)
pln  0.9256 0.9399)
pri  0.6824 0.7800)
rod  0.7255 0.7863)
vrl  0.9153 0.9418)
vrt  0.9275 0.9498)


In [ ]:
# The original DS is very imbalanced, that means the model would not learn the rare classes well.
# he baseline Random Forest scored F1=0.72 which is not bad, but that is because of imbalance.
# After removing the outliers and generating synthetic data, the f1 jumped to 78 and CV f1 is 79 which is not overfitting.
# arc jumped from 56 to 73 and phg jumped from 62 to 80. 
# I had a limitation plm is zero for both, that is because smotenc with k=3 neighbors can not  create meaningful synthetic data for plm.

# Is the optimized data realistic?
# Row sums are still average 1, which is expected for codon frequencies.
# Correlation structure similarity = 0.96 which means the relationship between codons did not get damaged.

# What is my limitation? 
# As I mentioned before, plm class is zero for both.
# 78 is not a bad  but not very high 
# map method actually did not help me, that is because continuous features  are already bonded in 0-1 range.
